In [46]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable
import re

# =========================================================
# CONFIG
# =========================================================
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

BRONZE_PATH = "Files/UA_TEST"

LONG_FILE = f"{BRONZE_PATH}/kpi_actual_long.csv"
WIDE_FILE = f"{BRONZE_PATH}/kpi_actual_wide.csv"
KPI_FILE  = f"{BRONZE_PATH}/kpi_master_dim.csv"

spark = (
    SparkSession.builder
    .appName("kpi_actuals_bronze_to_silver")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)

# =========================================================
# READ DATAFRAMES
# =========================================================
df_kpi_long_raw = (
    spark.read
         .option("header", True)
         .option("inferSchema", False)
         .csv(LONG_FILE)
)

df_kpi_wide_raw = (
    spark.read
         .option("header", True)
         .option("inferSchema", False)
         .csv(WIDE_FILE)
)

df_kpi_dim_raw = (
    spark.read
         .option("header", True)
         .option("inferSchema", False)
         .csv(KPI_FILE)
)

# =========================================================
# DISPLAY
# =========================================================
display(df_kpi_long_raw)
display(df_kpi_wide_raw)
display(df_kpi_dim_raw)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e07ccb65-f8a2-4a2c-bc7a-2dff99d9dfe2)

SynapseWidget(Synapse.DataFrame, a7fcc4b7-5db0-4cd5-8644-0868cffd3156)

SynapseWidget(Synapse.DataFrame, 43fb7474-34ce-4af4-8682-9cde3431ac78)

Typos


In [3]:
# =========================================================
# ADD ROW_ID + SOURCE_FILE
# =========================================================
df_kpi_long_step1 = (
    df_kpi_long_raw
    .withColumn(
        "ROW_ID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Department").cast("string"), F.lit("")),
                F.coalesce(F.col("KPI_Name").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_KPI_Type").cast("string"), F.lit("")),
                F.coalesce(F.col("Unit").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("SOURCE_FILE", F.lit("kpi_actual_long.csv"))
)

df_kpi_wide_step1 = (
    df_kpi_wide_raw
    .withColumn(
        "ROW_ID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Department").cast("string"), F.lit("")),
                F.coalesce(F.col("KPI_Name").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_KPI_Type").cast("string"), F.lit("")),
                F.coalesce(F.col("Unit").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("SOURCE_FILE", F.lit("kpi_actual_wide.csv"))
)

# =========================================================
# IDENTIFY VALUE COLUMNS
# - long: M1..M12 + Target if exists
# - wide: Jan-24..Dec-24 + Target if exists
# =========================================================
long_value_cols = [c for c in df_kpi_long_step1.columns if re.fullmatch(r"M([1-9]|1[0-2])", c)]
wide_value_cols = [c for c in df_kpi_wide_step1.columns if re.fullmatch(r"[A-Za-z]{3}-\d{2}", c)]

if "Target" in df_kpi_long_step1.columns:
    long_value_cols.append("Target")

if "Target" in df_kpi_wide_step1.columns:
    wide_value_cols.append("Target")

print("long value cols:", long_value_cols)
print("wide value cols:", wide_value_cols)

# =========================================================
# OCR FIX FUNCTION
# =========================================================
def fix_ocr_o_to_zero(df, value_cols):
    log_dfs = []
    out_df = df

    for col_name in value_cols:
        before_col = f"__before_{col_name}"
        after_col = f"__after_{col_name}"

        out_df = out_df.withColumn(before_col, F.col(col_name))

        out_df = out_df.withColumn(
            after_col,
            F.when(
                F.col(col_name).isNotNull() & F.col(col_name).rlike(r"^[0-9Oo\.\,\-\s]+$"),
                F.regexp_replace(F.col(col_name), r"[Oo]", "0")
            ).otherwise(F.col(col_name))
        )

        log_df = (
            out_df
            .filter(
                F.coalesce(F.col(before_col).cast("string"), F.lit("")) !=
                F.coalesce(F.col(after_col).cast("string"), F.lit(""))
            )
            .select(
                "ROW_ID",
                "SOURCE_FILE",
                F.lit(col_name).alias("COLUMN_NAME"),
                F.col(before_col).cast("string").alias("ORIGINAL_VALUE"),
                F.col(after_col).cast("string").alias("CORRECTED_VALUE"),
                F.current_timestamp().alias("LOGGED_AT")
            )
        )

        log_dfs.append(log_df)

        out_df = out_df.withColumn(col_name, F.col(after_col))
        out_df = out_df.drop(before_col, after_col)

    if log_dfs:
        final_log = log_dfs[0]
        for x in log_dfs[1:]:
            final_log = final_log.unionByName(x)
    else:
        final_log = spark.createDataFrame(
            [],
            "ROW_ID string, SOURCE_FILE string, COLUMN_NAME string, ORIGINAL_VALUE string, CORRECTED_VALUE string, LOGGED_AT timestamp"
        )

    return out_df, final_log

# =========================================================
# APPLY OCR FIX
# =========================================================
df_kpi_long_ocr_fixed, df_kpi_long_ocr_log = fix_ocr_o_to_zero(df_kpi_long_step1, long_value_cols)
df_kpi_wide_ocr_fixed, df_kpi_wide_ocr_log = fix_ocr_o_to_zero(df_kpi_wide_step1, wide_value_cols)

df_kpi_ocr_log = df_kpi_long_ocr_log.unionByName(df_kpi_wide_ocr_log)

# =========================================================
# PREVIEW
# =========================================================
print("=== OCR LOG ===")
display(df_kpi_ocr_log)

print("=== LONG AFTER OCR FIX ===")
display(df_kpi_long_ocr_fixed)

print("=== WIDE AFTER OCR FIX ===")
display(df_kpi_wide_ocr_fixed)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 5, Finished, Available, Finished, False)

long value cols: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'Target']
wide value cols: ['Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'Target']
=== OCR LOG ===


SynapseWidget(Synapse.DataFrame, adb1456b-3743-4122-a4d5-283a08f48e52)

=== LONG AFTER OCR FIX ===


SynapseWidget(Synapse.DataFrame, 688a2c78-53aa-4b2b-8ff5-be0faebcefa3)

=== WIDE AFTER OCR FIX ===


SynapseWidget(Synapse.DataFrame, 57b77419-4995-4ca2-b9f6-14596aec5dcf)

In [4]:
from pyspark.sql import functions as F

# =========================================================
# DEFINE CATEGORICAL COLUMNS
# =========================================================
categorical_cols = [
    "Pillar",
    "Sub_Pillar",
    "Department",
    "KPI_Name",
    "Sub_KPI_Type",
    "Unit"
]

# =========================================================
# ENSURE ROW_ID + SOURCE_FILE EXIST
# =========================================================
if "ROW_ID" not in df_kpi_long_ocr_fixed.columns:
    df_kpi_long_ocr_fixed = df_kpi_long_ocr_fixed.withColumn(
        "ROW_ID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Department").cast("string"), F.lit("")),
                F.coalesce(F.col("KPI_Name").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_KPI_Type").cast("string"), F.lit("")),
                F.coalesce(F.col("Unit").cast("string"), F.lit(""))
            ),
            256
        )
    )

if "SOURCE_FILE" not in df_kpi_long_ocr_fixed.columns:
    df_kpi_long_ocr_fixed = df_kpi_long_ocr_fixed.withColumn("SOURCE_FILE", F.lit("kpi_actual_long.csv"))

if "ROW_ID" not in df_kpi_wide_ocr_fixed.columns:
    df_kpi_wide_ocr_fixed = df_kpi_wide_ocr_fixed.withColumn(
        "ROW_ID",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_Pillar").cast("string"), F.lit("")),
                F.coalesce(F.col("Department").cast("string"), F.lit("")),
                F.coalesce(F.col("KPI_Name").cast("string"), F.lit("")),
                F.coalesce(F.col("Sub_KPI_Type").cast("string"), F.lit("")),
                F.coalesce(F.col("Unit").cast("string"), F.lit(""))
            ),
            256
        )
    )

if "SOURCE_FILE" not in df_kpi_wide_ocr_fixed.columns:
    df_kpi_wide_ocr_fixed = df_kpi_wide_ocr_fixed.withColumn("SOURCE_FILE", F.lit("kpi_actual_wide.csv"))

# =========================================================
# NORMALIZATION RULE
# =========================================================
def normalize_categorical_value(col_name):
    cleaned = F.trim(F.regexp_replace(F.col(col_name), r"\s+", " "))
    
    if col_name in ["Pillar"]:
        return F.upper(cleaned)
    else:
        return cleaned

# =========================================================
# FUNCTION: NORMALIZE + LOG EVERY CHANGE
# =========================================================
def normalize_and_log(df, categorical_columns):
    log_dfs = []
    cleaned_df = df

    for col_name in categorical_columns:
        if col_name not in cleaned_df.columns:
            continue

        before_col = f"__before_{col_name}"
        after_col = f"__after_{col_name}"

        cleaned_df = cleaned_df.withColumn(before_col, F.col(col_name))
        cleaned_df = cleaned_df.withColumn(after_col, normalize_categorical_value(col_name))

        log_df = (
            cleaned_df
            .filter(
                F.coalesce(F.col(before_col).cast("string"), F.lit("")) !=
                F.coalesce(F.col(after_col).cast("string"), F.lit(""))
            )
            .select(
                "ROW_ID",
                "SOURCE_FILE",
                F.lit(col_name).alias("COLUMN_NAME"),
                F.col(before_col).cast("string").alias("ORIGINAL_VALUE"),
                F.col(after_col).cast("string").alias("CORRECTED_VALUE"),
                F.current_timestamp().alias("LOGGED_AT")
            )
        )

        log_dfs.append(log_df)

        cleaned_df = cleaned_df.withColumn(col_name, F.col(after_col))
        cleaned_df = cleaned_df.drop(before_col, after_col)

    if log_dfs:
        final_log = log_dfs[0]
        for x in log_dfs[1:]:
            final_log = final_log.unionByName(x)
    else:
        final_log = spark.createDataFrame(
            [],
            "ROW_ID string, SOURCE_FILE string, COLUMN_NAME string, ORIGINAL_VALUE string, CORRECTED_VALUE string, LOGGED_AT timestamp"
        )

    return cleaned_df, final_log

# =========================================================
# APPLY TO LONG AND WIDE
# =========================================================
df_kpi_long_cleaned, df_kpi_long_norm_log = normalize_and_log(df_kpi_long_ocr_fixed, categorical_cols)
df_kpi_wide_cleaned, df_kpi_wide_norm_log = normalize_and_log(df_kpi_wide_ocr_fixed, categorical_cols)

# combine logs
df_kpi_norm_log = df_kpi_long_norm_log.unionByName(df_kpi_wide_norm_log)

# =========================================================
# PREVIEW
# =========================================================
print("=== NORMALIZATION LOG ===")
display(df_kpi_norm_log)

print("=== LONG AFTER NORMALIZATION ===")
display(df_kpi_long_cleaned)

print("=== WIDE AFTER NORMALIZATION ===")
display(df_kpi_wide_cleaned)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 6, Finished, Available, Finished, False)

=== NORMALIZATION LOG ===


SynapseWidget(Synapse.DataFrame, 245ff2f8-5374-4ab1-9bc3-59312f6cf561)

=== LONG AFTER NORMALIZATION ===


SynapseWidget(Synapse.DataFrame, 3084c214-0d3a-4e7f-8883-430048fc4a30)

=== WIDE AFTER NORMALIZATION ===


SynapseWidget(Synapse.DataFrame, ee1f0d0c-e9b8-4f88-9987-5fb9a39fe82c)

In [23]:
from pyspark.sql import Window
from pyspark.sql import functions as F
import re

# =========================================================
# BUSINESS KEY COLUMNS
# =========================================================
business_key_cols = [
    "Pillar",
    "Sub_Pillar",
    "Department",
    "KPI_Name",
    "Sub_KPI_Type",
    "Unit"
]

# =========================================================
# VALUE COLUMNS FOR EACH SOURCE
# =========================================================
long_month_cols = [c for c in df_kpi_long_cleaned.columns if re.fullmatch(r"M([1-9]|1[0-2])", c)]
wide_month_cols = [c for c in df_kpi_wide_cleaned.columns if re.fullmatch(r"[A-Za-z]{3}-\d{2}", c)]

long_value_cols = long_month_cols + ([c for c in ["Target"] if c in df_kpi_long_cleaned.columns])
wide_value_cols = wide_month_cols + ([c for c in ["Target"] if c in df_kpi_wide_cleaned.columns])

long_dedup_cols = business_key_cols + long_value_cols
wide_dedup_cols = business_key_cols + wide_value_cols

print("long dedup cols:", long_dedup_cols)
print("wide dedup cols:", wide_dedup_cols)


# =========================================================
# FUNCTION: DEDUP + LOG
# =========================================================
def deduplicate_and_log(df, dedup_cols, source_name):
    """
    Keep first row per duplicate group.
    Drop remaining rows and log them.

    Uses monotonically_increasing_id() as tiebreaker in the Window orderBy
    because all duplicate rows share the same ROW_ID (SHA256 of business keys).
    Without a tiebreaker, orderBy(ROW_ID) is non-deterministic — Spark may pick
    a different "first" row on each run.
    """

    # Add a stable tiebreaker so row ordering is deterministic across runs
    df_with_seq = df.withColumn("__row_seq", F.monotonically_increasing_id())

    w = Window.partitionBy(
        *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in dedup_cols]
    ).orderBy("__row_seq")

    df_ranked = (
        df_with_seq
        .withColumn("DUP_RANK", F.row_number().over(w))
        .withColumn(
            "IS_DUPLICATE",
            F.when(F.col("DUP_RANK") > 1, F.lit(True)).otherwise(F.lit(False))
        )
    )

    # Rows to keep — drop internal columns
    df_deduped = (
        df_ranked
        .filter(F.col("IS_DUPLICATE") == False)
        .drop("DUP_RANK", "__row_seq")
    )

    # Rows dropped
    df_dropped = df_ranked.filter(F.col("IS_DUPLICATE") == True)

    # Log dropped rows
    dedup_reason = (
        f"Dropped as duplicate within {source_name} "
        f"after OCR fix and categorical normalization"
    )

    dedup_key_expr = F.concat_ws(
        " || ",
        *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in dedup_cols]
    )

    df_dedup_log = (
        df_dropped
        .select(
            F.lit("DUPLICATE_DROP").alias("LOG_TYPE"),
            "ROW_ID",
            "SOURCE_FILE",
            F.lit("ROW_LEVEL").alias("COLUMN_NAME"),
            F.lit(None).cast("string").alias("ORIGINAL_VALUE"),
            dedup_key_expr.alias("CORRECTED_VALUE"),
            F.lit(dedup_reason).alias("DETAILS"),
            F.current_timestamp().alias("LOGGED_AT")
        )
    )

    # Summary log
    dropped_count = df_dropped.count()
    kept_count = df_deduped.count()

    summary_log = (
        spark.createDataFrame(
            [(
                "DUPLICATE_SUMMARY",
                source_name,
                dropped_count,
                kept_count,
                f"Detected duplicates using columns: {', '.join(dedup_cols)}"
            )],
            ["LOG_TYPE", "SOURCE_FILE", "DROPPED_COUNT", "KEPT_COUNT", "DETAILS"]
        )
        .withColumn("LOGGED_AT", F.current_timestamp())
    )

    return df_deduped, df_dedup_log, summary_log


# =========================================================
# APPLY TO LONG AND WIDE
# =========================================================
df_kpi_long_deduped, df_kpi_long_dedup_log, df_kpi_long_dedup_summary = deduplicate_and_log(
    df_kpi_long_cleaned,
    long_dedup_cols,
    "kpi_actual_long.csv"
)

df_kpi_wide_deduped, df_kpi_wide_dedup_log, df_kpi_wide_dedup_summary = deduplicate_and_log(
    df_kpi_wide_cleaned,
    wide_dedup_cols,
    "kpi_actual_wide.csv"
)

# Combine row-level logs
df_kpi_dedup_log = df_kpi_long_dedup_log.unionByName(df_kpi_wide_dedup_log)

# Combine summary logs
df_kpi_dedup_summary = df_kpi_long_dedup_summary.unionByName(df_kpi_wide_dedup_summary)

# =========================================================
# PREVIEW
# =========================================================
print("=== DEDUP ROW-LEVEL LOG ===")
display(df_kpi_dedup_log)

print("=== DEDUP SUMMARY LOG ===")
display(df_kpi_dedup_summary)

print("=== LONG AFTER DEDUP ===")
display(df_kpi_long_deduped)

print("=== WIDE AFTER DEDUP ===")
display(df_kpi_wide_deduped)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 27, Finished, Available, Finished, False)

long dedup cols: ['Pillar', 'Sub_Pillar', 'Department', 'KPI_Name', 'Sub_KPI_Type', 'Unit', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'Target']
wide dedup cols: ['Pillar', 'Sub_Pillar', 'Department', 'KPI_Name', 'Sub_KPI_Type', 'Unit', 'Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'Target']
=== DEDUP ROW-LEVEL LOG ===


SynapseWidget(Synapse.DataFrame, c9325f44-851c-4dff-ad77-6524f2a9cdbd)

=== DEDUP SUMMARY LOG ===


SynapseWidget(Synapse.DataFrame, 61fd706b-147d-4aed-90c8-a4b0936a9e18)

=== LONG AFTER DEDUP ===


SynapseWidget(Synapse.DataFrame, 0b9b1d90-218b-4217-b822-22b5b601948a)

=== WIDE AFTER DEDUP ===


SynapseWidget(Synapse.DataFrame, 28a7c95e-bb95-4a37-847a-9092ddd7b49d)

Unpivot Wide


In [24]:
from pyspark.sql import functions as F
import re

# =========================================================
# UNPIVOT WIDE FILE (Jan-24...Dec-24) INTO PERIOD (YYYY-MM)
# =========================================================
def unpivot_wide(df):
    # Get all month columns (e.g., Jan-24, Feb-24, ..., Dec-24)
    month_cols = [col for col in df.columns if re.fullmatch(r"[A-Za-z]{3}-\d{2}", col)]
    
    # Apply the unpivot operation using a correctly formatted string for stack
    stack_expr = ", ".join([f"'{col}', `{col}`" for col in month_cols])
    unpivoted_df = (
        df.select(
            "*",  # Keep all existing columns
            F.expr(f"stack({len(month_cols)}, {stack_expr}) as (month_token, actual_value)")
        )
        .withColumn("PERIOD", F.concat(F.lit("20"), F.regexp_extract(F.col("month_token"), r"([A-Za-z]{3})-(\d{2})", 2), F.lit("-"), F.lit("01")))
        .drop("month_token")  # Drop the temporary 'month_token' column
    )

    return unpivoted_df

# =========================================================
# APPLY UNPIVOT TO WIDE DATA
# =========================================================
df_kpi_wide_unpivoted = unpivot_wide(df_kpi_wide_deduped)

# Preview the unpivoted dataframe
print("=== WIDE AFTER UNPIVOT ===")
display(df_kpi_wide_unpivoted)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 28, Finished, Available, Finished, False)

=== WIDE AFTER UNPIVOT ===


SynapseWidget(Synapse.DataFrame, 83a44f2c-18cc-42a1-a310-c064fb7cdd29)

Unpivot Long

In [31]:
# =========================================================
# UNPIVOT LONG FORMAT M1 - M12 INTO PERIOD (YYYY-MM)
# =========================================================
def unpivot_long(df):
    # Get all month columns (e.g., M1, M2, ..., M12)
    month_cols = [col for col in df.columns if re.fullmatch(r"M([1-9]|1[0-2])", col)]
    
    # Apply unpivot operation using stack to convert columns to rows
    stack_expr = ", ".join([f"'{col}', `{col}`" for col in month_cols])
    unpivoted_df = (
        df.select(
            "*",  # Keep all existing columns
            F.expr(f"stack({len(month_cols)}, {stack_expr}) as (month_token, actual_value)")  # unpivot months
        )
        .withColumn("PERIOD", F.concat(F.lit("2024-"), F.lpad(F.regexp_extract(F.col("month_token"), r"^M(\d{1,2})$", 1), 2, "0")))
        .drop("month_token")  # Drop the temporary 'month_token' column
    )

    return unpivoted_df

# =========================================================
# APPLY UNPIVOT TO LONG DATA
# =========================================================
df_kpi_long_unpivoted = unpivot_long(df_kpi_long_deduped)

# Preview the unpivoted dataframe
print("=== LONG AFTER UNPIVOT ===")
display(df_kpi_long_unpivoted)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 35, Finished, Available, Finished, False)

=== LONG AFTER UNPIVOT ===


SynapseWidget(Synapse.DataFrame, f320e425-df17-499e-adcf-81c3dc72c95a)

Recoil

In [45]:
from pyspark.sql import functions as F

# =========================================================
# RECONCILE BOTH SOURCES INTO A SINGLE UNIFIED SCHEMA
# =========================================================
def reconcile_sources(df_long, df_wide):
    # Normalize missing Sub_Pillar
    df_long = df_long.withColumn("Sub_Pillar", F.coalesce(F.col("Sub_Pillar"), F.lit("UNKNOWN")))
    df_wide = df_wide.withColumn("Sub_Pillar", F.coalesce(F.col("Sub_Pillar"), F.lit("UNKNOWN")))

    # Handle missing Sub_KPI_Type by replacing NULL or empty with 'N/A'
    df_long = df_long.withColumn("Sub_KPI_Type", F.coalesce(F.col("Sub_KPI_Type"), F.lit("N/A")))
    df_wide = df_wide.withColumn("Sub_KPI_Type", F.coalesce(F.col("Sub_KPI_Type"), F.lit("N/A")))

    # Ensure the columns match before union
    unified_df = df_long.unionByName(df_wide, allowMissingColumns=True)

    # Clean up discrepancies in Sub_Pillar
    unified_df = unified_df.withColumn("Sub_Pillar", F.trim(F.regexp_replace(F.col("Sub_Pillar"), r"\s+", " ")))

    # Optionally: Deduplicate based on key columns
    unified_df = unified_df.dropDuplicates(
        ["Pillar", "Sub_Pillar", "Department", "KPI_Name", "Sub_KPI_Type", "Unit", "PERIOD"]
    )

    return unified_df


# =========================================================
# APPLY RECONCILE TO LONG AND WIDE UNPIVOTED DATA
# =========================================================
df_kpi_unified = reconcile_sources(df_kpi_long_unpivoted, df_kpi_wide_unpivoted)
df_kpi_unified = df_kpi_unified.select(
    "Pillar", 
    "Sub_Pillar", 
    "Department", 
    "KPI_Name", 
    "Sub_KPI_Type", 
    "Unit", 
    "Target",  # Ensure the column exists in df_raw
    "SOURCE_FILE", 
    "actual_value", 
    "PERIOD", 
    "IS_DUPLICATE",
)
# Preview the unified dataframe
print("=== UNIFIED DATA AFTER RECONCILE ===")
display(df_kpi_unified)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 49, Finished, Available, Finished, False)

=== UNIFIED DATA AFTER RECONCILE ===


SynapseWidget(Synapse.DataFrame, 6b3a3e1c-7b72-4174-81ab-b29ccc95cea1)

In [56]:
from pyspark.sql import functions as F

df_kpi_dim_raw = df_kpi_dim_raw.withColumn(
    "Sub_KPI_Type", F.coalesce(F.col("Sub_KPI_Type"), F.lit("N/A"))
)

df_kpi_dim_raw = df_kpi_dim_raw.filter(F.col("Is_Active") == "Y")

# Perform the join with aliases for the columns to avoid ambiguity
df_kpi_with_ua_id = df_kpi_unified.alias("df_raw").join(
    df_kpi_dim_raw.alias("df_unified"),
    (F.col("df_raw.Pillar") == F.col("df_unified.Pillar")) &
    (F.col("df_raw.Sub_Pillar") == F.col("df_unified.Sub_Pillar")) &
    (F.col("df_raw.Department") == F.col("df_unified.Department")) &
    (F.col("df_raw.KPI_Name") == F.col("df_unified.KPI_Name")) &
    (F.col("df_raw.Sub_KPI_Type") == F.col("df_unified.Sub_KPI_Type")),
    how="left"  # Left join to ensure no data loss from df_kpi
)
df_kpi_with_ua_id = df_kpi_with_ua_id.withColumn(
    "IS_ORPHANED",
    F.when(F.col("UA_ID").isNull(), True).otherwise(False)
)
# Debugging: Print schema to ensure the Target column is present
df_kpi_with_ua_id.printSchema()
df_kpi_with_ua_id = df_kpi_with_ua_id.select(
    "PILLAR_ID",
    "SP_ID",
    "UA_ID",
    "Yearly_Target",
    "Aggregation_Method",
    "df_raw.*",
    "Is_Active",
    "Valid_From",
    "Valid_To",	
    "Source_Version",
    "IS_ORPHANED"
)
# Display the result
print("=== Result after join with UA_ID ===")
display(df_kpi_with_ua_id)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 60, Finished, Available, Finished, False)

root
 |-- Pillar: string (nullable = true)
 |-- Sub_Pillar: string (nullable = false)
 |-- Department: string (nullable = true)
 |-- KPI_Name: string (nullable = true)
 |-- Sub_KPI_Type: string (nullable = false)
 |-- Unit: string (nullable = true)
 |-- Target: string (nullable = true)
 |-- SOURCE_FILE: string (nullable = false)
 |-- actual_value: string (nullable = true)
 |-- PERIOD: string (nullable = true)
 |-- IS_DUPLICATE: boolean (nullable = true)
 |-- PILLAR_ID: string (nullable = true)
 |-- SP_ID: string (nullable = true)
 |-- UA_ID: string (nullable = true)
 |-- Pillar: string (nullable = true)
 |-- Sub_Pillar: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- KPI_Name: string (nullable = true)
 |-- Sub_KPI_Type: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Yearly_Target: string (nullable = true)
 |-- Aggregation_Method: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Valid_From: string (nullable = true)
 |-- Va

SynapseWidget(Synapse.DataFrame, 37441c15-1773-4e1d-aa89-7d964357c405)

In [58]:
hash_cols = [
    "PILLAR_ID", "SP_ID", "UA_ID",
    "Pillar", "Sub_Pillar", "Department",
    "KPI_Name", "Sub_KPI_Type", "Unit",
    "PERIOD", "actual_value"
]

df_silver_kpi = df_kpi_with_ua_id.withColumn(
    "ROW_HASH",
    F.sha2(
        F.concat_ws("||", *[
            F.coalesce(F.col(c).cast("string"), F.lit(""))
            for c in hash_cols
        ]),
        256
    )
)
display(df_silver_kpi)

StatementMeta(, 207dac8d-861f-43bf-932f-07ec9263a9be, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0294433f-a7b7-4ef8-90d7-c50a73c40ac5)